In [1]:
# =============================================================================
# RESEARCHER CLUSTERING — CONCEPT FEATURES ONLY
# Dataset : researchers_final_1.csv
# Features: concepts + concept_scores  (NO one-hot encoding)
# Models  : TF-IDF, Weighted TF-IDF, NMF, LDA, SBERT
#           × MiniBatchKMeans, BisectingKMeans, Birch, HDBSCAN
# Outputs : per-model  → metrics CSV, PNG graph, PKL pipeline, JSON profiles
#           final      → comparison PNG + ranking CSV
# ~370k rows  →  N_CLUSTERS = 2500
# =============================================================================

In [2]:
import os, gc, re, ast, json, time, pickle, psutil, warnings, traceback
from datetime import datetime
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # save-only backend (swap to TkAgg for interactive)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans, Birch, BisectingKMeans
from sklearn.decomposition   import NMF, TruncatedSVD, LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing   import normalize
from sklearn.metrics import (silhouette_score,davies_bouldin_score,calinski_harabasz_score)
from sklearn.neighbors import KNeighborsClassifier
from scipy.sparse import issparse
import hdbscan
from sentence_transformers import SentenceTransformer

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (16, 6)

# CONFIG

In [3]:
CSV_PATH         = "./researchers_final_1.csv"
OUTPUT_DIR       = "./clustered_outputs"
RANDOM_STATE     = 42
N_CLUSTERS       = 3000       # ~370k / 2500 ≈ 148 per cluster
REDUCED_DIM      = 100        # SVD / NMF / LDA components
MAX_FEATURES     = 15000      # TF-IDF / CountVec vocabulary cap
MAX_NOISE_THRESH = 0.20       # 20 % → WARNING for HDBSCAN
SBERT_BATCH_SIZE = 1024       # lower to 256 if RAM is tight
KMEANS_BATCH     = 4096

os.makedirs(OUTPUT_DIR, exist_ok=True)
timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results  = []
perf_metrics = []
error_log    = []

print(f"Config loaded | timestamp={timestamp} | output_dir={OUTPUT_DIR}")

Config loaded | timestamp=20260219_200925 | output_dir=./clustered_outputs


# UTILITIES

In [4]:
def get_memory_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def clear_memory(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

# CONCEPT PARSING

In [5]:
def parse_list_column(val):
    """Parse concepts column → list of strings regardless of storage format."""
    if isinstance(val, list):
        return [str(x).strip() for x in val if str(x).strip()]
    if pd.isna(val) or str(val).strip() in ("", "[]", "nan", "NaN"):
        return []
    s = str(val).strip()
    if s.startswith("["):
        for parser in (ast.literal_eval, json.loads):
            try:
                return [str(x).strip() for x in parser(s) if str(x).strip()]
            except Exception:
                pass
    parts = re.split(r"[|,;]+", s.strip("[]"))
    return [p.strip().strip("'\"") for p in parts if p.strip().strip("'\"")]

def parse_scores_column(val):
    """Parse concept_scores column → list of floats."""
    if isinstance(val, (list, np.ndarray)):
        return [float(x) for x in val]
    if pd.isna(val) or str(val).strip() in ("", "[]", "nan", "NaN"):
        return []
    s = str(val).strip()
    if s.startswith("["):
        try:
            return [float(x) for x in ast.literal_eval(s)]
        except Exception:
            pass
    parts = re.split(r"[|,;\s]+", s.strip("[]"))
    result = []
    for p in parts:
        try: result.append(float(p.strip()))
        except: pass
    return result

def concepts_to_text(concepts):
    return " ".join(c.lower().replace(" ", "_") for c in concepts)

def weighted_concepts_to_text(concepts, scores, repeat_max=5):
    """Repeat each concept proportional to its score (0-1 → 1-repeat_max times)."""
    if not concepts:
        return ""
    if not scores or len(scores) != len(concepts):
        return concepts_to_text(concepts)
    tokens = []
    for c, s in zip(concepts, scores):
        term    = c.lower().replace(" ", "_")
        repeats = max(1, min(repeat_max, round(float(s) * repeat_max)))
        tokens.extend([term] * repeats)
    return " ".join(tokens)

In [6]:
# EVALUATION

In [7]:
def evaluate_clusters(X, labels, model_name, mem_before, mem_after, exec_time):
    """Compute Silhouette, Davies-Bouldin, Calinski-Harabasz with safe subsampling."""
    try:
        mask       = labels != -1
        n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
        noise_cnt  = int((labels == -1).sum())
        noise_pct  = noise_cnt / len(labels) * 100

        sil, db, ch = -1.0, 999.0, 0.0

        if mask.sum() >= 10 and n_clusters >= 2:
            X_m, l_m = X[mask], labels[mask]
            if X_m.shape[0] > 15000:
                idx  = np.random.RandomState(RANDOM_STATE).choice(X_m.shape[0], 15000, replace=False)
                X_m, l_m = X_m[idx], l_m[idx]
            if issparse(X_m):
                X_m = X_m.toarray()

            try:  sil = float(silhouette_score(X_m, l_m, sample_size=min(10000, len(l_m))))
            except Exception as e:
                error_log.append({"model": model_name, "metric": "silhouette", "error": str(e)})

            try:  db  = float(davies_bouldin_score(X_m, l_m))
            except Exception as e:
                error_log.append({"model": model_name, "metric": "davies_bouldin", "error": str(e)})

            try:  ch  = float(calinski_harabasz_score(X_m, l_m))
            except Exception as e:
                error_log.append({"model": model_name, "metric": "calinski_harabasz", "error": str(e)})

        status = "SUCCESS"
        if n_clusters < 10:
            status = "FAILED - Too Few Clusters"
        elif noise_pct > MAX_NOISE_THRESH * 100:
            status = "WARNING - High Noise"

        metrics = {
            "model":                   model_name,
            "status":                  status,
            "n_clusters":              int(n_clusters),
            "noise_count":             noise_cnt,
            "noise_percentage":        round(noise_pct, 2),
            "silhouette_score":        round(sil, 4),
            "davies_bouldin_index":    round(db, 4),
            "calinski_harabasz_score": round(ch, 2),
            "memory_used_mb":          round(mem_after - mem_before, 2),
            "execution_time_sec":      round(exec_time, 2),
            "memory_before_mb":        round(mem_before, 2),
            "memory_after_mb":         round(mem_after, 2),
        }
        perf_metrics.append(metrics)

        print(f"  ✓ Status: {status}")
        print(f"    Clusters: {n_clusters} | Noise: {noise_pct:.2f}%")
        print(f"    Silhouette: {sil:.4f} | Davies-Bouldin: {db:.4f} | Calinski-Harabasz: {ch:.2f}")
        print(f"    Memory Δ: {mem_after - mem_before:.1f} MB | Time: {exec_time:.2f}s")

        return labels, metrics

    except Exception as e:
        print(f"  ✗ Evaluation error: {e}")
        error_log.append({"model": model_name, "phase": "evaluation", "error": str(e)})
        return labels, None

# PLOTTING (per-model)

In [8]:
def plot_model_metrics(metrics_dict, model_name, output_dir):
    """Bar chart of the three clustering metrics — save + display."""
    try:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f"Clustering Metrics — {model_name}", fontsize=14, fontweight="bold")

        configs = [
            ("silhouette_score",        "Silhouette Score\n(↑ better)",        "steelblue"),
            ("davies_bouldin_index",    "Davies-Bouldin Index\n(↓ better)",    "tomato"),
            ("calinski_harabasz_score", "Calinski-Harabasz Score\n(↑ better)", "seagreen"),
        ]

        for ax, (key, title, color) in zip(axes, configs):
            val  = metrics_dict.get(key, 0)
            bars = ax.bar([model_name], [val], color=color, alpha=0.80, edgecolor="black")
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.tick_params(axis="x", rotation=20, labelsize=8)
            ax.grid(axis="y", alpha=0.4)
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h,
                        f"{h:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

        info = (f"Clusters: {metrics_dict.get('n_clusters','?')}  |  "
                f"Noise: {metrics_dict.get('noise_percentage',0):.2f}%  |  "
                f"Time: {metrics_dict.get('execution_time_sec',0):.1f}s  |  "
                f"Memory Δ: {metrics_dict.get('memory_used_mb',0):.1f} MB")
        fig.text(0.5, -0.02, info, ha="center", fontsize=9, style="italic", color="gray")

        plt.tight_layout()
        path = os.path.join(output_dir, f"{model_name}_metrics.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        # plt.show()   # uncomment for interactive display
        plt.close()
        print(f"  💾 Graph saved: {model_name}_metrics.png")

    except Exception as e:
        print(f"  ⚠️  Graph error: {e}")
        plt.close()

In [9]:
# SAVING OUTPUTS (metrics CSV, PKL, JSON)

In [10]:
def build_cluster_profiles(df_full, labels, top_n=10):
    """Per-cluster summary: size, avg numeric stats, top concepts."""
    profiles = {}
    tmp = df_full.copy()
    tmp["_cluster"] = labels

    numeric_cols = [c for c in
                    ["works_count", "cited_by_count", "h_index",
                     "i10_index", "citation_velocity"]
                    if c in tmp.columns]

    for cid in sorted(set(int(x) for x in labels)):
        sub     = tmp[tmp["_cluster"] == cid]
        profile = {"cluster_id": cid, "size": len(sub)}

        for col in numeric_cols:
            try:    profile[f"avg_{col}"] = round(float(sub[col].mean()), 2)
            except: profile[f"avg_{col}"] = None

        cnt = Counter()
        for clist in sub["_parsed_concepts"]:
            cnt.update(clist)
        profile["top_concepts"] = [c for c, _ in cnt.most_common(top_n)]
        profiles[cid] = profile

    return profiles

In [11]:
def save_model_outputs(model_name, labels, metrics, df_full,
                        pipeline, output_dir, ts):
    """
    Saves:
      {model_name}_metrics_{ts}.csv   — performance metrics only (1 row)
      {model_name}_{ts}.pkl           — full pipeline for web-app prediction
      {model_name}_{ts}.json          — cluster profiles + web-app guide
    """
    try:
        # ── 1. Metrics CSV (performance metrics only) ────────────────────────
        csv_path = os.path.join(output_dir, f"{model_name}_metrics_{ts}.csv")
        pd.DataFrame([metrics]).to_csv(csv_path, index=False)
        print(f"  💾 Metrics CSV: {model_name}_metrics_{ts}.csv")

        # ── 2. PKL — pipeline for prediction ────────────────────────────────
        pkl_data = {
            "model_name": model_name,
            "timestamp":  ts,
            "metrics":    metrics,
            "n_samples":  int(len(labels)),
            "pipeline":   pipeline,   # vectorizer + reducer + clusterer
            "labels":     labels.tolist() if hasattr(labels, "tolist") else list(labels),
        }
        pkl_path = os.path.join(output_dir, f"{model_name}_{ts}.pkl")
        with open(pkl_path, "wb") as f:
            pickle.dump(pkl_data, f)
        print(f"  💾 PKL:         {model_name}_{ts}.pkl")

        # ── 3. JSON — cluster profiles + web-app guide ───────────────────────
        profiles  = build_cluster_profiles(df_full, labels)
        json_data = {
            "model_name":          model_name,
            "timestamp":           ts,
            "metrics":             metrics,
            "n_samples":           int(len(labels)),
            "unique_clusters":     int(len(set(labels))),
            "cluster_distribution": {
                int(k): int(v)
                for k, v in pd.Series(labels).value_counts().items()
            },
            "cluster_profiles":    profiles,
            "web_app_usage": {
                "description": (
                    "Load the matching .pkl file. "
                    "Transform input concepts through pipeline, "
                    "call clusterer.predict() to get cluster_id, "
                    "then look up cluster_profiles[cluster_id] for cluster data."
                ),
                "input_format": "List of concept strings, e.g. ['Machine Learning', 'Deep Learning']",
                "output":       "{'cluster_id': int, 'cluster_profile': {...}}",
            },
        }
        json_path = os.path.join(output_dir, f"{model_name}_{ts}.json")
        with open(json_path, "w") as f:
            json.dump(json_data, f, indent=2, cls=NumpyEncoder)
        print(f"  💾 JSON:        {model_name}_{ts}.json")

    except Exception as e:
        print(f"  ⚠️  Save error [{model_name}]: {e}")
        error_log.append({"model": model_name, "phase": "save_outputs",
                           "error": str(e), "traceback": traceback.format_exc()})

In [12]:
# FINAL COMPARISON PLOT

In [13]:
def plot_comparison_graph(metrics_df, output_dir, ts):
    """Horizontal bar comparison of all models on the 3 core metrics."""
    try:
        n = len(metrics_df)
        fig, axes = plt.subplots(1, 3, figsize=(21, max(5, n * 0.55 + 2)))
        fig.suptitle("Model Comparison — Clustering Performance",
                     fontsize=16, fontweight="bold")

        configs = [
            ("silhouette_score",        "Silhouette Score (↑ better)",        "steelblue",  True),
            ("davies_bouldin_index",    "Davies-Bouldin Index (↓ better)",    "tomato",     False),
            ("calinski_harabasz_score", "Calinski-Harabasz Score (↑ better)", "seagreen",   True),
        ]

        for ax, (metric, title, color, hi_better) in zip(axes, configs):
            sorted_df = metrics_df.sort_values(metric, ascending=not hi_better)
            bars = ax.barh(sorted_df["model"], sorted_df[metric],
                           color=color, alpha=0.75, edgecolor="black")
            ax.set_title(title, fontsize=12, fontweight="bold")
            ax.grid(axis="x", alpha=0.35)
            ax.tick_params(axis="y", labelsize=9)
            for bar in bars:
                w = bar.get_width()
                ax.text(w, bar.get_y() + bar.get_height() / 2,
                        f" {w:.3f}", ha="left", va="center", fontsize=8)

        plt.tight_layout()
        path = os.path.join(output_dir, f"model_comparison_{ts}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        # plt.show()   # uncomment for interactive display
        plt.close()
        print(f"💾 Comparison graph saved: model_comparison_{ts}.png")

    except Exception as e:
        print(f"⚠️  Comparison graph error: {e}")
        plt.close()

# DATA LOADING & FEATURE ENGINEERING

In [14]:
print("\n" + "=" * 70)
print("DATA LOADING")
print("=" * 70)

df = pd.read_csv(CSV_PATH, low_memory=False).drop_duplicates().reset_index(drop=True)

for col in ["concepts", "concept_scores"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print(f"✓ Loaded {len(df):,} rows")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")
print(f"✓ Memory:  {get_memory_usage():.1f} MB")

# Parse concepts + scores
df["_parsed_concepts"] = df["concepts"].apply(parse_list_column)
df["_parsed_scores"]   = df["concept_scores"].apply(parse_scores_column)

# Build text representations
df["TEXT_CONCEPTS"] = df["_parsed_concepts"].apply(concepts_to_text).replace("", "unknown")
df["TEXT_WEIGHTED"] = df.apply(
    lambda r: weighted_concepts_to_text(r["_parsed_concepts"], r["_parsed_scores"]),
    axis=1
).replace("", "unknown")

print(f"✓ TEXT_CONCEPTS sample : {df['TEXT_CONCEPTS'].iloc[0][:80]}")
print(f"✓ TEXT_WEIGHTED sample : {df['TEXT_WEIGHTED'].iloc[0][:80]}")
print(f"✓ Empty concept rows   : {(df['TEXT_CONCEPTS'] == 'unknown').sum():,}")


DATA LOADING
✓ Loaded 386,584 rows
✓ Columns: openalex_id, name, orcid, works_count, cited_by_count, h_index, i10_index, citation_velocity, concepts, concept_scores, first_publication_year, last_publication_year
✓ Memory:  658.6 MB
✓ TEXT_CONCEPTS sample : doppler_effect physics calibration optics materials_science
✓ TEXT_WEIGHTED sample : doppler_effect doppler_effect doppler_effect physics physics calibration calibra
✓ Empty concept rows   : 98


In [15]:
# MODEL 1 — TF-IDF + MiniBatchKMeans

In [16]:
print("MODEL 1: TF-IDF (concepts) + MiniBatchKMeans")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_CONCEPTS"])
    svd    = TruncatedSVD(n_components=REDUCED_DIM, random_state=RANDOM_STATE)
    X      = normalize(svd.fit_transform(mat), norm="l2")

    km     = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                              batch_size=KMEANS_BATCH, n_init=5, max_iter=150)
    lbl    = km.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "TFIDF_MiniBatchKMeans", mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "TFIDF_MiniBatchKMeans", OUTPUT_DIR)
        save_model_outputs("TFIDF_MiniBatchKMeans", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": svd, "clusterer": km,
                            "feature_type": "tfidf", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("TFIDF_MiniBatchKMeans", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "TFIDF_MiniBatchKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 1: TF-IDF (concepts) + MiniBatchKMeans
  ✓ Status: SUCCESS
    Clusters: 2959 | Noise: 0.00%
    Silhouette: 0.3479 | Davies-Bouldin: 1.0627 | Calinski-Harabasz: 68.79
    Memory Δ: 711.5 MB | Time: 114.45s
  💾 Graph saved: TFIDF_MiniBatchKMeans_metrics.png
  💾 Metrics CSV: TFIDF_MiniBatchKMeans_metrics_20260219_200925.csv
  💾 PKL:         TFIDF_MiniBatchKMeans_20260219_200925.pkl
  💾 JSON:        TFIDF_MiniBatchKMeans_20260219_200925.json


# MODEL 2 — TF-IDF + BisectingKMeans

In [17]:
print("MODEL 2: TF-IDF (concepts) + BisectingKMeans")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_CONCEPTS"])
    svd    = TruncatedSVD(n_components=REDUCED_DIM, random_state=RANDOM_STATE)
    X      = normalize(svd.fit_transform(mat), norm="l2")

    bk     = BisectingKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                               n_init=3, max_iter=150)
    lbl    = bk.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "TFIDF_BisectingKMeans", mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "TFIDF_BisectingKMeans", OUTPUT_DIR)
        save_model_outputs("TFIDF_BisectingKMeans", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": svd, "clusterer": bk,
                            "feature_type": "tfidf", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("TFIDF_BisectingKMeans", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "TFIDF_BisectingKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 2: TF-IDF (concepts) + BisectingKMeans
  ✓ Status: SUCCESS
    Clusters: 3000 | Noise: 0.00%
    Silhouette: 0.2982 | Davies-Bouldin: 1.2185 | Calinski-Harabasz: 60.23
    Memory Δ: 17.5 MB | Time: 68.71s
  💾 Graph saved: TFIDF_BisectingKMeans_metrics.png
  💾 Metrics CSV: TFIDF_BisectingKMeans_metrics_20260219_200925.csv
  💾 PKL:         TFIDF_BisectingKMeans_20260219_200925.pkl
  💾 JSON:        TFIDF_BisectingKMeans_20260219_200925.json


# MODEL 3 — TF-IDF + Birch

In [18]:
print("MODEL 3: TF-IDF (concepts) + Birch")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_CONCEPTS"])
    svd    = TruncatedSVD(n_components=REDUCED_DIM, random_state=RANDOM_STATE)
    X      = normalize(svd.fit_transform(mat), norm="l2")

    birch  = Birch(n_clusters=N_CLUSTERS, threshold=0.4, branching_factor=100)
    lbl    = birch.fit_predict(X)

    # KNN for robust web-app prediction
    knn    = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X, lbl)

    lbl, m = evaluate_clusters(X, lbl, "TFIDF_Birch", mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "TFIDF_Birch", OUTPUT_DIR)
        save_model_outputs("TFIDF_Birch", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": svd,
                            "clusterer": birch, "knn": knn,
                            "feature_type": "tfidf", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("TFIDF_Birch", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "TFIDF_Birch", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 3: TF-IDF (concepts) + Birch
  ✓ Status: SUCCESS
    Clusters: 3000 | Noise: 0.00%
    Silhouette: 0.2871 | Davies-Bouldin: 0.9613 | Calinski-Harabasz: 61.67
    Memory Δ: 33.2 MB | Time: 79.94s
  💾 Graph saved: TFIDF_Birch_metrics.png
  💾 Metrics CSV: TFIDF_Birch_metrics_20260219_200925.csv
  💾 PKL:         TFIDF_Birch_20260219_200925.pkl
  💾 JSON:        TFIDF_Birch_20260219_200925.json


# MODEL 4 — TF-IDF + HDBSCAN  (noise remapped via KNN)

In [19]:
print("MODEL 4: TF-IDF (concepts) + HDBSCAN  [noise → KNN remap]")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_CONCEPTS"])
    svd    = TruncatedSVD(n_components=REDUCED_DIM, random_state=RANDOM_STATE)
    X      = normalize(svd.fit_transform(mat), norm="l2")

    hdb    = hdbscan.HDBSCAN(min_cluster_size=80, min_samples=15,
                               metric="euclidean", cluster_selection_method="eom",
                               core_dist_n_jobs=4)
    raw_lbl = hdb.fit_predict(X)

    valid   = raw_lbl != -1
    knn     = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X[valid], raw_lbl[valid])
    lbl     = knn.predict(X)   # all noise points assigned to nearest cluster

    lbl, m  = evaluate_clusters(X, lbl, "TFIDF_HDBSCAN", mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "TFIDF_HDBSCAN", OUTPUT_DIR)
        save_model_outputs("TFIDF_HDBSCAN", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": svd,
                            "clusterer": hdb, "knn": knn,
                            "feature_type": "tfidf", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("TFIDF_HDBSCAN", lbl))
    clear_memory(mat, X, hdb)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "TFIDF_HDBSCAN", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 4: TF-IDF (concepts) + HDBSCAN  [noise → KNN remap]
  ✓ Status: SUCCESS
    Clusters: 1297 | Noise: 0.00%
    Silhouette: 0.2636 | Davies-Bouldin: 1.3552 | Calinski-Harabasz: 77.80
    Memory Δ: -1833.9 MB | Time: 62620.33s
  💾 Graph saved: TFIDF_HDBSCAN_metrics.png
  💾 Metrics CSV: TFIDF_HDBSCAN_metrics_20260219_200925.csv
  💾 PKL:         TFIDF_HDBSCAN_20260219_200925.pkl
  💾 JSON:        TFIDF_HDBSCAN_20260219_200925.json


# MODEL 5 — Weighted TF-IDF (concept_scores) + MiniBatchKMeans

In [20]:
print("MODEL 5: Weighted TF-IDF (concept_scores weighted) + MiniBatchKMeans")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_WEIGHTED"])
    svd    = TruncatedSVD(n_components=REDUCED_DIM, random_state=RANDOM_STATE)
    X      = normalize(svd.fit_transform(mat), norm="l2")

    km     = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                              batch_size=KMEANS_BATCH, n_init=5, max_iter=150)
    lbl    = km.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "WeightedTFIDF_MiniBatchKMeans",
                                mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "WeightedTFIDF_MiniBatchKMeans", OUTPUT_DIR)
        save_model_outputs("WeightedTFIDF_MiniBatchKMeans", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": svd, "clusterer": km,
                            "feature_type": "weighted_tfidf",
                            "input_col": "TEXT_WEIGHTED"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("WeightedTFIDF_MiniBatchKMeans", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "WeightedTFIDF_MiniBatchKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 5: Weighted TF-IDF (concept_scores weighted) + MiniBatchKMeans
  ✓ Status: SUCCESS
    Clusters: 2994 | Noise: 0.00%
    Silhouette: 0.3448 | Davies-Bouldin: 0.9859 | Calinski-Harabasz: 112.64
    Memory Δ: 709.4 MB | Time: 149.03s
  💾 Graph saved: WeightedTFIDF_MiniBatchKMeans_metrics.png
  💾 Metrics CSV: WeightedTFIDF_MiniBatchKMeans_metrics_20260219_200925.csv
  💾 PKL:         WeightedTFIDF_MiniBatchKMeans_20260219_200925.pkl
  💾 JSON:        WeightedTFIDF_MiniBatchKMeans_20260219_200925.json


# MODEL 6 — NMF + MiniBatchKMeans

In [21]:
print("MODEL 6: NMF (concepts) + MiniBatchKMeans")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    tfidf  = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2),
                              min_df=3, max_df=0.85, sublinear_tf=True)
    mat    = tfidf.fit_transform(df["TEXT_CONCEPTS"])
    nmf    = NMF(n_components=REDUCED_DIM, random_state=RANDOM_STATE,
                  max_iter=200, init="nndsvda")
    X      = normalize(nmf.fit_transform(mat), norm="l2")

    km     = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                              batch_size=KMEANS_BATCH, n_init=5, max_iter=150)
    lbl    = km.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "NMF_MiniBatchKMeans",
                                mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "NMF_MiniBatchKMeans", OUTPUT_DIR)
        save_model_outputs("NMF_MiniBatchKMeans", lbl, m, df,
                           {"vectorizer": tfidf, "reducer": nmf, "clusterer": km,
                            "feature_type": "nmf", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("NMF_MiniBatchKMeans", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "NMF_MiniBatchKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 6: NMF (concepts) + MiniBatchKMeans
  ✓ Status: SUCCESS
    Clusters: 2969 | Noise: 0.00%
    Silhouette: 0.3508 | Davies-Bouldin: 1.0404 | Calinski-Harabasz: 119.25
    Memory Δ: -1738.8 MB | Time: 16855.27s
  💾 Graph saved: NMF_MiniBatchKMeans_metrics.png
  💾 Metrics CSV: NMF_MiniBatchKMeans_metrics_20260219_200925.csv
  💾 PKL:         NMF_MiniBatchKMeans_20260219_200925.pkl
  💾 JSON:        NMF_MiniBatchKMeans_20260219_200925.json


# MODEL 7 — LDA + MiniBatchKMeans

In [22]:
print("MODEL 7: LDA (concepts) + MiniBatchKMeans")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    cv   = CountVectorizer(max_features=MAX_FEATURES, min_df=3,
                           max_df=0.85, ngram_range=(1,2))
    mat  = cv.fit_transform(df["TEXT_CONCEPTS"])
    lda  = LatentDirichletAllocation(n_components=REDUCED_DIM,
                                      random_state=RANDOM_STATE,
                                      max_iter=25, learning_method="online",
                                      batch_size=KMEANS_BATCH, n_jobs=4)
    X    = normalize(lda.fit_transform(mat), norm="l2")

    km   = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                            batch_size=KMEANS_BATCH, n_init=5, max_iter=150)
    lbl  = km.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "LDA_MiniBatchKMeans",
                                mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "LDA_MiniBatchKMeans", OUTPUT_DIR)
        save_model_outputs("LDA_MiniBatchKMeans", lbl, m, df,
                           {"vectorizer": cv, "reducer": lda, "clusterer": km,
                            "feature_type": "lda", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("LDA_MiniBatchKMeans", lbl))
    clear_memory(mat, X)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "LDA_MiniBatchKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 7: LDA (concepts) + MiniBatchKMeans
  ✓ Status: SUCCESS
    Clusters: 2899 | Noise: 0.00%
    Silhouette: 0.3302 | Davies-Bouldin: 1.3406 | Calinski-Harabasz: 69.95
    Memory Δ: -854.3 MB | Time: 1889.30s
  💾 Graph saved: LDA_MiniBatchKMeans_metrics.png
  💾 Metrics CSV: LDA_MiniBatchKMeans_metrics_20260219_200925.csv
  💾 PKL:         LDA_MiniBatchKMeans_20260219_200925.pkl
  💾 JSON:        LDA_MiniBatchKMeans_20260219_200925.json


# MODEL 8 — SBERT + MiniBatchKMeans

In [23]:
print("MODEL 8: SBERT (all-MiniLM-L6-v2) + MiniBatchKMeans")
print(f"  ~{len(df):,} rows — expect 20–60 min on CPU")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    sbert  = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    texts  = df["TEXT_CONCEPTS"].tolist()
    n_bat  = (len(texts) + SBERT_BATCH_SIZE - 1) // SBERT_BATCH_SIZE
    embs   = []
    for i in range(0, len(texts), SBERT_BATCH_SIZE):
        embs.append(sbert.encode(texts[i:i+SBERT_BATCH_SIZE],
                                  show_progress_bar=False,
                                  convert_to_numpy=True))
        b = i // SBERT_BATCH_SIZE
        if b % 50 == 0:
            print(f"  Batch {b+1}/{n_bat} | Mem: {get_memory_usage():.0f} MB")

    X   = normalize(np.vstack(embs), norm="l2")
    km  = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                           batch_size=KMEANS_BATCH, n_init=5, max_iter=150)
    lbl = km.fit_predict(X)

    lbl, m = evaluate_clusters(X, lbl, "SBERT_MiniBatchKMeans",
                                mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "SBERT_MiniBatchKMeans", OUTPUT_DIR)
        save_model_outputs("SBERT_MiniBatchKMeans", lbl, m, df,
                           {"sbert_model": sbert, "clusterer": km,
                            "feature_type": "sbert", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("SBERT_MiniBatchKMeans", lbl))
    clear_memory(X, embs)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "SBERT_MiniBatchKMeans", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 8: SBERT (all-MiniLM-L6-v2) + MiniBatchKMeans
  ~386,584 rows — expect 20–60 min on CPU
  Batch 1/378 | Mem: 1433 MB
  Batch 51/378 | Mem: 1522 MB
  Batch 101/378 | Mem: 1592 MB
  Batch 151/378 | Mem: 1664 MB
  Batch 201/378 | Mem: 1755 MB
  Batch 251/378 | Mem: 1817 MB
  Batch 301/378 | Mem: 1158 MB
  Batch 351/378 | Mem: 1238 MB
  ✓ Status: SUCCESS
    Clusters: 2963 | Noise: 0.00%
    Silhouette: 0.3206 | Davies-Bouldin: 1.2234 | Calinski-Harabasz: 43.72
    Memory Δ: 175.4 MB | Time: 2849.36s
  💾 Graph saved: SBERT_MiniBatchKMeans_metrics.png
  💾 Metrics CSV: SBERT_MiniBatchKMeans_metrics_20260219_200925.csv
  💾 PKL:         SBERT_MiniBatchKMeans_20260219_200925.pkl
  💾 JSON:        SBERT_MiniBatchKMeans_20260219_200925.json


# MODEL 9 — SBERT + HDBSCAN  (noise remapped via KNN)

In [24]:
print("MODEL 9: SBERT (all-MiniLM-L6-v2) + HDBSCAN  [noise → KNN remap]")
print(f"  ~{len(df):,} rows — may take several hours on CPU")
try:
    mem_b = get_memory_usage(); t0 = time.time()

    sbert  = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    texts  = df["TEXT_CONCEPTS"].tolist()
    n_bat  = (len(texts) + SBERT_BATCH_SIZE - 1) // SBERT_BATCH_SIZE
    embs   = []
    for i in range(0, len(texts), SBERT_BATCH_SIZE):
        embs.append(sbert.encode(texts[i:i+SBERT_BATCH_SIZE],
                                  show_progress_bar=False,
                                  convert_to_numpy=True))
        b = i // SBERT_BATCH_SIZE
        if b % 50 == 0:
            print(f"  Batch {b+1}/{n_bat} | Mem: {get_memory_usage():.0f} MB")

    X      = normalize(np.vstack(embs), norm="l2")
    hdb    = hdbscan.HDBSCAN(min_cluster_size=80, min_samples=15,
                               metric="euclidean", cluster_selection_method="eom",
                               core_dist_n_jobs=4)
    raw_lbl = hdb.fit_predict(X)

    valid   = raw_lbl != -1
    knn     = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X[valid], raw_lbl[valid])
    lbl     = knn.predict(X)

    lbl, m  = evaluate_clusters(X, lbl, "SBERT_HDBSCAN",
                                 mem_b, get_memory_usage(), time.time()-t0)
    if m:
        plot_model_metrics(m, "SBERT_HDBSCAN", OUTPUT_DIR)
        save_model_outputs("SBERT_HDBSCAN", lbl, m, df,
                           {"sbert_model": sbert, "clusterer": hdb, "knn": knn,
                            "feature_type": "sbert", "input_col": "TEXT_CONCEPTS"},
                           OUTPUT_DIR, timestamp)
    all_results.append(("SBERT_HDBSCAN", lbl))
    clear_memory(X, embs, hdb)

except Exception as e:
    print(f"✗ ERROR: {e}")
    error_log.append({"model": "SBERT_HDBSCAN", "error": str(e),
                       "traceback": traceback.format_exc()})

MODEL 9: SBERT (all-MiniLM-L6-v2) + HDBSCAN  [noise → KNN remap]
  ~386,584 rows — may take several hours on CPU
  Batch 1/378 | Mem: 1766 MB
  Batch 51/378 | Mem: 1856 MB
  Batch 101/378 | Mem: 1906 MB
  Batch 151/378 | Mem: 1947 MB
  Batch 201/378 | Mem: 1314 MB
  Batch 251/378 | Mem: 636 MB
  Batch 301/378 | Mem: 684 MB
  Batch 351/378 | Mem: 722 MB
  ✓ Status: SUCCESS
    Clusters: 1384 | Noise: 0.00%
    Silhouette: 0.2335 | Davies-Bouldin: 1.5431 | Calinski-Harabasz: 49.54
    Memory Δ: -2275.2 MB | Time: 235546.08s
  💾 Graph saved: SBERT_HDBSCAN_metrics.png
  💾 Metrics CSV: SBERT_HDBSCAN_metrics_20260219_200925.csv
  💾 PKL:         SBERT_HDBSCAN_20260219_200925.pkl
  💾 JSON:        SBERT_HDBSCAN_20260219_200925.json


# FINAL COMPARISON

In [25]:
print("FINAL COMPARISON")
try:
    metrics_df = pd.DataFrame(perf_metrics)

    # All-models metrics CSV
    all_csv = os.path.join(OUTPUT_DIR, f"all_models_metrics_{timestamp}.csv")
    metrics_df.to_csv(all_csv, index=False)
    print(f"💾 All-models metrics CSV: all_models_metrics_{timestamp}.csv")

    # Comparison graph
    plot_comparison_graph(metrics_df, OUTPUT_DIR, timestamp)

    # Filter valid
    valid = metrics_df[
        (metrics_df["silhouette_score"] > 0) &
        (metrics_df["davies_bouldin_index"] < 999) &
        (metrics_df["n_clusters"] >= 50)
    ].copy()

    print(f"\n📊 Total models run:  {len(metrics_df)}")
    print(f"✓  Valid models:      {len(valid)}")
    print(f"✗  Excluded:          {len(metrics_df) - len(valid)}")

    if len(valid) > 0:
        def norm_col(col, ascending=True):
            mn, mx = col.min(), col.max()
            if mx == mn:
                return pd.Series(0.5, index=col.index)
            n = (col - mn) / (mx - mn)
            return n if ascending else 1 - n

        valid["sil_n"]   = norm_col(valid["silhouette_score"],        True)
        valid["db_n"]    = norm_col(valid["davies_bouldin_index"],     False)
        valid["ch_n"]    = norm_col(valid["calinski_harabasz_score"],  True)
        valid["noise_n"] = 1 - valid["noise_percentage"] / 100

        valid["composite"] = (
            0.40 * valid["sil_n"] +
            0.35 * valid["db_n"]  +
            0.15 * valid["ch_n"]  +
            0.10 * valid["noise_n"]
        )

        ranked = valid.sort_values("composite", ascending=False)

        print("\nRANKING (all valid models)\n" + "─" * 70)
        for rank, (_, row) in enumerate(ranked.iterrows(), 1):
            print(f"{rank:2}. {row['model']}")
            print(f"    Composite: {row['composite']:.4f}  |  "
                  f"Silhouette: {row['silhouette_score']:.4f}  |  "
                  f"DBI: {row['davies_bouldin_index']:.4f}  |  "
                  f"CHI: {row['calinski_harabasz_score']:.2f}")
            print(f"    Clusters: {row['n_clusters']}  |  "
                  f"Noise: {row['noise_percentage']:.2f}%  |  "
                  f"Time: {row['execution_time_sec']:.1f}s\n")

        ranking_csv = os.path.join(OUTPUT_DIR, f"model_ranking_{timestamp}.csv")
        ranked[["model", "composite", "silhouette_score", "davies_bouldin_index",
                "calinski_harabasz_score", "n_clusters", "noise_percentage",
                "execution_time_sec", "memory_used_mb"]
        ].to_csv(ranking_csv, index=False)
        print(f"💾 Ranking CSV: model_ranking_{timestamp}.csv")

        best = ranked.iloc[0]
        print(f"\n🏆 BEST MODEL: {best['model']}")
        print(f"   Composite Score:   {best['composite']:.4f}")
        print(f"   Silhouette Score:  {best['silhouette_score']:.4f}")
        print(f"   Davies-Bouldin:    {best['davies_bouldin_index']:.4f}")
        print(f"   Calinski-Harabasz: {best['calinski_harabasz_score']:.2f}")
        print(f"   Clusters:          {best['n_clusters']}")

except Exception as e:
    print(f"✗ Final analysis error: {e}")
    error_log.append({"phase": "final_analysis", "error": str(e),
                       "traceback": traceback.format_exc()})

if error_log:
    err_csv = os.path.join(OUTPUT_DIR, f"error_log_{timestamp}.csv")
    pd.DataFrame(error_log).to_csv(err_csv, index=False)
    print(f"\n⚠️  {len(error_log)} errors logged → error_log_{timestamp}.csv")

FINAL COMPARISON
💾 All-models metrics CSV: all_models_metrics_20260219_200925.csv
💾 Comparison graph saved: model_comparison_20260219_200925.png

📊 Total models run:  9
✓  Valid models:      9
✗  Excluded:          0

RANKING (all valid models)
──────────────────────────────────────────────────────────────────────
 1. NMF_MiniBatchKMeans
    Composite: 0.9524  |  Silhouette: 0.3508  |  DBI: 1.0404  |  CHI: 119.25
    Clusters: 2969  |  Noise: 0.00%  |  Time: 16855.3s

 2. WeightedTFIDF_MiniBatchKMeans
    Composite: 0.9516  |  Silhouette: 0.3448  |  DBI: 0.9859  |  CHI: 112.64
    Clusters: 2994  |  Noise: 0.00%  |  Time: 149.0s

 3. TFIDF_MiniBatchKMeans
    Composite: 0.8289  |  Silhouette: 0.3479  |  DBI: 1.0627  |  CHI: 68.79
    Clusters: 2959  |  Noise: 0.00%  |  Time: 114.5s

 4. TFIDF_Birch
    Composite: 0.6684  |  Silhouette: 0.2871  |  DBI: 0.9613  |  CHI: 61.67
    Clusters: 3000  |  Noise: 0.00%  |  Time: 79.9s

 5. LDA_MiniBatchKMeans
    Composite: 0.6037  |  Silhouette: